In [ ]:
# 1. Data prep
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import xgboost as xgb

from gbnet import xgbmodule as xgm


XGB_PARAMS = {
    "eta": 1.0,
    "max_depth": 2,
    "lambda": 0.0,
    "alpha": 0.0,
    "gamma": 0.0,
    "min_child_weight": 0.0,
    "tree_method": "hist",
    "nthread": 1,
}


def make_fixture(n_rows=32, heldout_offset=100.0):
    X = np.arange(n_rows, dtype=np.float32).reshape(-1, 1)
    y = np.arange(n_rows, dtype=np.float32)
    y[1::2] = heldout_offset * np.where((np.arange(n_rows // 2) % 2) == 0, 1.0, -1.0)
    row_indices = np.arange(0, n_rows, 2, dtype=np.int64)
    heldout_rows = np.setdiff1d(np.arange(n_rows, dtype=np.int64), row_indices)
    return X, y.reshape(-1, 1), row_indices, heldout_rows


def native_xgb_predictions(X, y, row_indices, params):
    full_dtrain = xgb.DMatrix(X, label=y.reshape(-1))
    subset_dtrain = full_dtrain.slice(row_indices)
    booster = xgb.train(
        {**params, "objective": "reg:squarederror", "base_score": 0.0},
        subset_dtrain,
        num_boost_round=1,
    )
    return booster.predict(xgb.DMatrix(X)).reshape(-1, 1)


def gbnet_xgb_predictions(X, y, params, row_indices=None):
    module = xgm.XGBModule(
        batch_size=X.shape[0],
        input_dim=X.shape[1],
        output_dim=1,
        params=params,
    )
    loss_fn = torch.nn.MSELoss()

    module.train()
    preds = module(X)
    if row_indices is None:
        loss = 0.5 * loss_fn(preds, torch.tensor(y, dtype=torch.float))
        step_kwargs = {}
    else:
        loss = 0.5 * loss_fn(preds[row_indices], torch.tensor(y[row_indices], dtype=torch.float))
        step_kwargs = {"row_indices": row_indices}
    loss.backward(create_graph=True)
    module.gb_step(**step_kwargs)

    module.eval()
    return module(X).detach().numpy()


X, y, row_indices, heldout_rows = make_fixture()
comparison = pd.DataFrame({
    "x": X.reshape(-1),
    "y": y.reshape(-1),
    "is_subset_row": np.isin(np.arange(len(X)), row_indices),
})
comparison.head()


In [ ]:
# 2. Fixed row subsetting: GBNet vs native XGBoost DMatrix slicing
native_subset_preds = native_xgb_predictions(X, y, row_indices, XGB_PARAMS)
native_full_preds = native_xgb_predictions(X, y, np.arange(len(X), dtype=np.int64), XGB_PARAMS)
gbnet_subset_preds = gbnet_xgb_predictions(X, y, XGB_PARAMS, row_indices=row_indices)

comparison = comparison.assign(
    native_subset_pred=native_subset_preds.reshape(-1),
    native_full_pred=native_full_preds.reshape(-1),
    gbnet_subset_pred=gbnet_subset_preds.reshape(-1),
    subset_vs_full_abs_diff=np.abs(native_subset_preds.reshape(-1) - native_full_preds.reshape(-1)),
    gbnet_vs_native_abs_diff=np.abs(gbnet_subset_preds.reshape(-1) - native_subset_preds.reshape(-1)),
)

np.testing.assert_allclose(gbnet_subset_preds, native_subset_preds, rtol=1e-6, atol=1e-6)
assert comparison["subset_vs_full_abs_diff"].max() > 1e-3
assert np.abs(comparison.loc[heldout_rows, "native_subset_pred"]).max() > 1e-3

comparison[[
    "x",
    "y",
    "is_subset_row",
    "native_subset_pred",
    "native_full_pred",
    "gbnet_subset_pred",
    "subset_vs_full_abs_diff",
    "gbnet_vs_native_abs_diff",
]]


In [ ]:
# 3. Random subsetting: mini-batch-style rounds with timing
timing_rows = 4000
subset_size = 256
n_rounds = 30
rng = np.random.default_rng(0)

X_timing = np.arange(timing_rows, dtype=np.float32).reshape(-1, 1)
y_timing = (X_timing[:, 0] / 40.0 + np.sin(X_timing[:, 0] / 25.0)).astype(np.float32).reshape(-1, 1)
all_rows = np.arange(timing_rows, dtype=np.int64)
loss_fn = torch.nn.MSELoss()


def run_timed_rounds(row_sampler):
    module = xgm.XGBModule(
        batch_size=X_timing.shape[0],
        input_dim=X_timing.shape[1],
        output_dim=1,
        params=XGB_PARAMS,
    )
    round_times = []
    subset_sizes = []

    for _ in range(n_rounds):
        row_indices = row_sampler()

        start = time.perf_counter()
        module.train()
        module.zero_grad()
        preds = module(X_timing)
        loss = 0.5 * loss_fn(preds[row_indices], torch.tensor(y_timing[row_indices], dtype=torch.float))
        loss.backward(create_graph=True)
        module.gb_step(row_indices=row_indices)
        round_times.append(time.perf_counter() - start)
        subset_sizes.append(len(row_indices))

    return pd.DataFrame({
        "round": np.arange(1, n_rounds + 1),
        "seconds": round_times,
        "subset_size": subset_sizes,
    })


full_timing = run_timed_rounds(lambda: all_rows).assign(schedule="full data")
random_subset_timing = run_timed_rounds(
    lambda: np.sort(rng.choice(all_rows, size=subset_size, replace=False))
).assign(schedule="random subset")
timing = pd.concat([full_timing, random_subset_timing], ignore_index=True)

display(timing.groupby("schedule")[["seconds", "subset_size"]].agg(["mean", "min", "max"]))

fig, ax = plt.subplots(figsize=(9, 4))
for schedule, frame in timing.groupby("schedule"):
    ax.plot(frame["round"], frame["seconds"], marker="o", linewidth=1.5, label=schedule)
ax.set_title("Per-round XGBoost update time")
ax.set_xlabel("Boosting round")
ax.set_ylabel("seconds")
ax.legend()
ax.grid(alpha=0.3)
plt.show()
